# IntentRank — Intelligent Candidate Discovery Engine
**India Runs Hackathon · Data & AI Challenge · Track 1**
**Team: Apex Climbers**

This notebook builds a candidate ranking system layer by layer. Each section explains
*what* the layer does and *why* it exists, before the code that implements it.

**Pipeline overview:**
1. Load and inspect the dataset
2. Define what the job actually requires (JD signal extraction)
3. Honeypot and bad-data detection
4. Hard filters (eliminate disqualified candidates before scoring)
5. Semantic technical alignment scoring (sentence-transformers)
6. Skill depth scoring (quality over keyword count)
7. Behavioral availability scoring (Redrob signals)
8. Career trajectory scoring (direction of movement, not just current title)
9. JD-specific fit scoring (experience band, location, GitHub)
10. Composite scoring and ranking
11. Evidence-grounded reasoning generation
12. Final output and validation


## 1. Setup and Imports

We use `sentence-transformers` for real semantic embeddings of job descriptions and
candidate profiles — this is what lets the system understand *meaning*, not just
keyword overlap. A fast, CPU-friendly model (`all-MiniLM-L6-v2`) is used because the
hackathon constraint is **no GPU, under 5 minutes for the full candidate pool**.

If `sentence-transformers` is not available in your environment, the code falls back
to TF-IDF cosine similarity automatically — slightly weaker semantic understanding,
but it keeps the pipeline runnable everywhere.

In [1]:
import json
import math
import csv
import random
import re
from datetime import datetime, date
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd  # FIX: moved from cell 16 to imports

# Try to use real sentence embeddings. Fall back to TF-IDF if unavailable.
USE_EMBEDDINGS = True
try:
    from sentence_transformers import SentenceTransformer
    EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2")  # ~80MB, CPU-friendly, fast
    print("Using sentence-transformers (all-MiniLM-L6-v2) for semantic matching.")
except ImportError:
    USE_EMBEDDINGS = False
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    print("sentence-transformers not found — falling back to TF-IDF cosine similarity.")

REFERENCE_DATE = date(2026, 6, 29)  # "today" for recency calculations


# ── Helper utilities ──────────────────────────────────────────────────────────

def _safe_float(val, default: float = 0.0) -> float:
    """Safely cast a value to float, handling None, strings, etc."""
    try:
        return float(val) if val is not None else default
    except (TypeError, ValueError):
        return default


c:\Users\reach\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7315.07it/s]


Using sentence-transformers (all-MiniLM-L6-v2) for semantic matching.


## 2. Load the Dataset

The candidate pool is a `.jsonl` file — one JSON object per line, one per candidate.
We load it fully into memory since 100K candidate records at this schema size fits
comfortably in RAM (a few hundred MB), and in-memory scoring is far faster than
re-reading from disk for every step.

In [2]:
# FIX: Use pathlib.Path for cross-platform compatibility + existence check
CANDIDATES_PATH = (
    Path("[PUB] India_runs_data_and_ai_challenge")
    / "India_runs_data_and_ai_challenge"
    / "candidates.jsonl"
)

if not CANDIDATES_PATH.exists():
    raise FileNotFoundError(
        f"Candidates file not found at {CANDIDATES_PATH}. "
        f"Please adjust the path or ensure the data folder is in the working directory."
    )

candidates = []
with open(CANDIDATES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            candidates.append(json.loads(line))

print(f"Loaded {len(candidates):,} candidates")
print(f"Sample candidate fields: {list(candidates[0].keys())}")


Loaded 100,000 candidates
Sample candidate fields: ['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals']


## 3. Define What the Job Actually Requires

Before scoring anyone, we need a precise model of the job itself. This is the
**job description (JD) decomposition** step — instead of treating the JD as one
blob of text, we separate it into:

- **Hard skills**: directly required, non-negotiable technical capabilities
- **Preferred skills**: nice-to-have, gives a candidate an edge but isn't required
- **Core concept text**: a dense paragraph of all the underlying concepts, used to
  build the semantic embedding that candidate profiles get compared against
- **Title signals**: job titles that genuinely indicate the right background
- **Disqualifying title signals**: titles that indicate the candidate is in a
  completely different career track (marketing, HR, accounting, etc.), used later
  to catch keyword-stuffed resumes
- **Consulting firms list**: used to flag candidates whose entire career is at
  service/staffing companies with no hands-on product engineering — a common
  pattern that keyword-only systems miss entirely

This decomposition is the difference between *keyword matching* and *actually
understanding what the role needs* — which is the explicit ask in the challenge brief.

In [3]:
JD_CORE_CONCEPT_TEXT = (
    "We are hiring for a role centered on embeddings, retrieval, and ranking systems. "
    "The ideal candidate has built and deployed semantic search or recommendation systems "
    "to production, using vector databases such as Pinecone, Weaviate, Qdrant, FAISS, or "
    "OpenSearch, and understands hybrid search combining dense and sparse retrieval. "
    "Experience with evaluation frameworks (NDCG, MRR, MAP, offline and online A/B testing) "
    "is highly valued. Familiarity with fine-tuning language models, LoRA or QLoRA, and "
    "applied NLP in a production environment is a strong plus. We want someone who has "
    "shipped real systems serving real users, not only academic or research experiments."
)

JD_HARD_SKILLS = {
    "embeddings", "sentence transformers", "sentence-transformers", "vector database",
    "vector db", "pinecone", "weaviate", "qdrant", "milvus", "opensearch",
    "elasticsearch", "faiss", "hybrid search", "retrieval", "ranking",
    "semantic search", "ndcg", "mrr", "map", "evaluation framework",
    "a/b test", "python", "production", "deployed",
}

JD_PREFERRED_SKILLS = {
    "fine-tuning", "fine tuning", "lora", "qlora", "peft", "learning to rank",
    "xgboost", "neural ranking", "nlp", "information retrieval",
    "recommendation", "distributed systems", "inference optimization",
    "open source", "open-source",
}

GOOD_TITLE_SIGNALS = {
    "machine learning", "ml engineer", "ai engineer", "data scientist",
    "nlp engineer", "research engineer", "applied scientist", "search engineer",
    "ranking engineer", "recommendation", "backend engineer", "software engineer",
    "platform engineer", "data engineer", "mlops", "senior engineer",
    "staff engineer", "founding engineer", "tech lead", "principal engineer",
}

BAD_PRIMARY_TITLES = {
    "marketing manager", "content writer", "graphic designer", "hr manager",
    "accountant", "sales executive", "customer support", "operations manager",
    "project manager", "mechanical engineer", "civil engineer", "business analyst",
}

CONSULTING_FIRMS = {
    "wipro", "infosys", "tcs", "accenture", "cognizant", "capgemini", "hcl",
    "tech mahindra", "mphasis", "hexaware", "ltimindtree", "mindtree",
    "l&t infotech",
}

# FIX: Pre-compiled word-boundary regex for consulting firm detection.
# Avoids substring false positives like "accenturexyz" matching "accenture".
_CF_PATTERN = re.compile(
    r'\b(?:' + '|'.join(re.escape(cf) for cf in CONSULTING_FIRMS) + r')\b',
    re.IGNORECASE
)


def _is_consulting_company(company_name: str) -> bool:
    """True if company name contains a known consulting firm as a whole word."""
    return bool(_CF_PATTERN.search(company_name))


AI_DEPTH_SKILLS = {
    "embeddings", "vector search", "semantic search", "dense retrieval",
    "fine-tuning llms", "fine tuning llms", "llm fine-tuning", "rag",
    "retrieval augmented generation", "nlp", "transformers", "bert",
    "sentence transformers", "pytorch", "tensorflow", "scikit-learn", "sklearn",
    "faiss", "pinecone", "weaviate", "qdrant", "chromadb", "langchain",
    "llamaindex", "openai", "anthropic", "information retrieval", "ranking",
    "recommendation systems", "evaluation", "ndcg", "a/b testing", "mlops",
    "hugging face", "huggingface",
}

print("JD decomposed into hard skills, preferred skills, and title signal sets.")
print(f"Hard skills: {len(JD_HARD_SKILLS)} | Preferred skills: {len(JD_PREFERRED_SKILLS)}")


JD decomposed into hard skills, preferred skills, and title signal sets.
Hard skills: 24 | Preferred skills: 15


## 4. Honeypot Detection

The challenge brief explicitly states the dataset contains ~80 "honeypot" profiles —
candidates with subtly impossible data designed to catch systems that don't actually
validate the data they're scoring. Submitting too many honeypots in the final top 100
results in disqualification, so this check runs *before* anything else.

We check for two patterns:

1. **Impossible skill claims**: a skill marked "advanced" or "expert" but with
   `duration_months == 0` makes no sense — you can't be an expert in something
   you've used for zero time. Similarly, claiming expert-level proficiency in
   10+ different skills simultaneously is a strong signal of fabricated data.

2. **Date math that doesn't add up**: if a career entry's `start_date` and
   `end_date` imply a duration very different from the `duration_months` field
   the candidate (or the data generator) claims, that's an internal inconsistency
   — a real profile wouldn't have this contradiction.

In [4]:
def is_honeypot(candidate: dict) -> bool:
    """Returns True if the candidate profile contains internally impossible data."""
    skills = candidate.get("skills", [])

    # Check 1: expert/advanced skill claimed with zero duration
    for sk in skills:
        if sk.get("proficiency") in ("advanced", "expert") and sk.get("duration_months", 1) == 0:
            return True

    # Check 2: implausibly many "expert" skills at once
    expert_count = sum(1 for s in skills if s.get("proficiency") == "expert")
    if expert_count >= 10:
        return True

    # Check 3: career date math contradiction
    for hist in candidate.get("career_history", []):
        start_s, end_s = hist.get("start_date"), hist.get("end_date")
        claimed = hist.get("duration_months", 0)
        if start_s and end_s and claimed > 0:
            try:
                start = datetime.strptime(start_s, "%Y-%m-%d").date()
                end = datetime.strptime(end_s, "%Y-%m-%d").date()
                actual_months = (end.year - start.year) * 12 + (end.month - start.month)
                if claimed > actual_months + 6:  # generous tolerance for rounding
                    return True
            except ValueError:
                pass

    return False


# Quick sanity check on a sample
honeypot_count = sum(1 for c in candidates[:5000] if is_honeypot(c))
print(f"Honeypots detected in first 5,000 candidates: {honeypot_count} ({honeypot_count/5000:.1%})")


Honeypots detected in first 5,000 candidates: 2 (0.0%)


## 5. Hard Filters — Eliminate Before Scoring

This is where we directly address the brief's instruction to look "beyond keyword
filters." Ironically, the way you beat a bad keyword filter isn't to remove all
filtering — it's to filter on the *right* signal.

Three exclusion rules, applied in order:

1. **Honeypots** — excluded outright (from step 4 above)
2. **Consulting-only, non-technical career** — if every single job in someone's
   history is at a known consulting/staffing firm *and* their current title is
   clearly non-technical (e.g. "Operations Manager"), they are not a fit for an
   embeddings/ranking engineering role no matter what's listed in their skills section
3. **Keyword stuffers** — this is the classic trap: someone whose *title* is
   completely unrelated (Content Writer, HR Manager) but whose *skills list* is
   packed with AI buzzwords with no depth behind them. We check: does this
   non-technical-titled candidate have at least 3 skills with real AI depth?
   If not, they're excluded — their skills list is decorative, not evidence of
   actual capability.

Filtering early like this also makes the pipeline faster: we only run the expensive
semantic scoring on candidates worth scoring.

In [5]:
def hard_filter(candidate: dict) -> tuple[bool, str]:
    """Returns (keep: bool, reason_if_excluded: str)"""
    if is_honeypot(candidate):
        return False, "honeypot"

    profile = candidate.get("profile", {})
    career = candidate.get("career_history", [])
    # FIX: Guard against None from profile.get("current_title")
    current_title = (profile.get("current_title") or "").lower()

    if career:
        # FIX: Use word-boundary matching instead of substring matching
        all_consulting = all(
            _is_consulting_company(h.get("company", ""))
            for h in career
        )
        is_bad_title = any(bt in current_title for bt in BAD_PRIMARY_TITLES)
        if all_consulting and is_bad_title:
            return False, "consulting-only, non-technical career"

    is_bad_title = any(bt in current_title for bt in BAD_PRIMARY_TITLES)
    if is_bad_title:
        skills_text = " ".join(s.get("name", "").lower() for s in candidate.get("skills", []))
        ai_skill_count = sum(1 for ds in AI_DEPTH_SKILLS if ds in skills_text)
        if ai_skill_count < 3:
            return False, f"non-technical title ({current_title}) with shallow AI skill listing"

    return True, ""


# Run the filter across the full dataset and report what gets excluded and why
exclusion_reasons = Counter()
kept = []
for c in candidates:
    keep, reason = hard_filter(c)
    if keep:
        kept.append(c)
    else:
        exclusion_reasons[reason] += 1

print(f"Total candidates: {len(candidates):,}")
print(f"Excluded: {len(candidates) - len(kept):,}")
for reason, count in exclusion_reasons.most_common():
    print(f"  - {reason}: {count:,}")
print(f"Remaining pool to score: {len(kept):,}")


Total candidates: 100,000
Excluded: 63,710
  - non-technical title (hr manager) with shallow AI skill listing: 5,071
  - non-technical title (business analyst) with shallow AI skill listing: 5,025
  - non-technical title (mechanical engineer) with shallow AI skill listing: 5,004
  - non-technical title (project manager) with shallow AI skill listing: 4,969
  - non-technical title (accountant) with shallow AI skill listing: 4,962
  - non-technical title (operations manager) with shallow AI skill listing: 4,941
  - non-technical title (content writer) with shallow AI skill listing: 4,925
  - non-technical title (customer support) with shallow AI skill listing: 4,922
  - non-technical title (sales executive) with shallow AI skill listing: 4,910
  - non-technical title (civil engineer) with shallow AI skill listing: 4,870
  - non-technical title (graphic designer) with shallow AI skill listing: 4,832
  - non-technical title (marketing manager) with shallow AI skill listing: 4,814
  - consu

## 6. Semantic Technical Alignment Score

This is the core "beyond keywords" layer. Instead of checking whether a candidate's
resume contains the literal string "vector database," we convert both the job
description and the candidate's profile text into **dense embedding vectors** —
numerical representations that capture *meaning*, not just words — and measure how
close they are using cosine similarity.

This means a candidate who writes "built a system to find semantically similar
documents using nearest-neighbour search over dense vectors" will score well against
a JD that says "vector database" and "semantic search," even though not a single
word matches exactly. That is the actual definition of understanding context over
keyword matching.

We weight a candidate's text before embedding it:
- **Current role gets 3x weight**, since it's the strongest signal of present-day capability
- **Roles longer than 12 months get 2x weight**, since sustained work signals real depth
- **Short-tenure roles get 1x weight**, since brief stints are weaker evidence

If `sentence-transformers` isn't available, we fall back to TF-IDF + cosine similarity,
which is weaker (literal term overlap with some statistical weighting) but still
functional and fast.

In [6]:
def build_candidate_text(candidate: dict) -> str:
    """Builds a single weighted text blob representing the candidate's profile."""
    profile = candidate.get("profile", {})
    career = candidate.get("career_history", [])
    skills = candidate.get("skills", [])

    parts = [profile.get("headline", ""), profile.get("summary", "")]

    for hist in career:
        weight = 3 if hist.get("is_current") else (2 if hist.get("duration_months", 0) > 12 else 1)
        desc = hist.get("description", "") + " " + hist.get("title", "")
        parts.extend([desc] * weight)

    for sk in skills:
        name = sk.get("name", "")
        if sk.get("proficiency") in ("advanced", "expert") and sk.get("duration_months", 0) > 6:
            parts.append(name + " " + name)  # double-weight high-confidence skills
        else:
            parts.append(name)

    return " ".join(parts).lower().strip()


if USE_EMBEDDINGS:
    # Real semantic embeddings — encode the JD once
    jd_embedding = EMBED_MODEL.encode([JD_CORE_CONCEPT_TEXT], normalize_embeddings=True)

    def score_technical_alignment_batch(candidate_texts: list[str]) -> np.ndarray:
        """Encodes a batch of candidate texts and returns cosine similarity to the JD."""
        non_empty_idx = [i for i, t in enumerate(candidate_texts) if t.strip()]
        sims = np.zeros(len(candidate_texts))
        if non_empty_idx:
            texts_to_encode = [candidate_texts[i] for i in non_empty_idx]
            embeddings = EMBED_MODEL.encode(
                texts_to_encode, normalize_embeddings=True, batch_size=64, show_progress_bar=False
            )
            sim_scores = embeddings @ jd_embedding.T  # cosine sim since both normalized
            for idx, score in zip(non_empty_idx, sim_scores.flatten()):
                sims[idx] = float(score)
        return sims

else:
    # FIX: Fit TF-IDF on the FULL kept corpus (not a random sample) to avoid
    # vocabulary mismatch and OOV issues during transform.
    corpus = [JD_CORE_CONCEPT_TEXT] + [build_candidate_text(c) for c in kept]
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=8000, sublinear_tf=True, min_df=2)
    vectorizer.fit(corpus)
    jd_vector = vectorizer.transform([JD_CORE_CONCEPT_TEXT])
    print(f"TF-IDF vectorizer fitted on {len(corpus):,} documents, vocab={len(vectorizer.vocabulary_):,}")

    def score_technical_alignment_batch(candidate_texts: list[str]) -> np.ndarray:
        cand_vectors = vectorizer.transform(candidate_texts)
        return cosine_similarity(jd_vector, cand_vectors).flatten()


# Quick test on first 20 kept candidates
test_texts = [build_candidate_text(c) for c in kept[:20]]
test_scores = score_technical_alignment_batch(test_texts)
print("Sample technical alignment scores (first 20 candidates):")
for c, s in zip(kept[:20], test_scores):
    print(f"  {c['candidate_id']} | {c['profile']['current_title']:30s} | sim={s:.3f}")


Sample technical alignment scores (first 20 candidates):
  CAND_0000001 | Backend Engineer               | sim=0.318
  CAND_0000010 | Data Engineer                  | sim=0.287
  CAND_0000011 | QA Engineer                    | sim=0.227
  CAND_0000014 | Frontend Engineer              | sim=0.319
  CAND_0000015 | Software Engineer              | sim=0.200
  CAND_0000018 | Frontend Engineer              | sim=0.336
  CAND_0000021 | Project Manager                | sim=0.276
  CAND_0000023 | Software Engineer              | sim=0.327
  CAND_0000025 | Frontend Engineer              | sim=0.194
  CAND_0000027 | DevOps Engineer                | sim=0.221
  CAND_0000031 | Recommendation Systems Engineer | sim=0.631
  CAND_0000032 | .NET Developer                 | sim=0.259
  CAND_0000035 | Full Stack Developer           | sim=0.279
  CAND_0000038 | Java Developer                 | sim=0.279
  CAND_0000043 | Cloud Engineer                 | sim=0.293
  CAND_0000044 | Frontend Engineer        

## 7. Skill Depth Score

A candidate listing 20 AI buzzwords with no further detail is not the same as a
candidate listing 5 AI skills they've used for 2+ years with verified endorsements.
This layer rewards **depth over breadth** — it's how we avoid promoting keyword
stuffers even after they've passed the hard filter (some keyword stuffers have
technical-sounding titles, so they won't get caught by the title-based filter).

For each skill that is genuinely relevant to the role (checked against `AI_DEPTH_SKILLS`),
we compute a depth contribution based on:

- **Proficiency level** (expert > advanced > intermediate > beginner)
- **Duration multiplier** — a skill used for 24+ months gets full credit; less than
  that scales down proportionally. A skill with `duration_months = 0` gets almost
  no credit even if labelled "expert" (this also softens any honeypots that slipped
  through the hard filter)
- **Endorsement bonus** — using a log scale so that going from 0→10 endorsements
  matters more than going from 100→110
- **Assessment score bonus** — Redrob's own skill assessment data, when available,
  is a stronger signal than self-reported proficiency since it's a tested measurement

In [7]:
def score_skill_depth(candidate: dict) -> float:
    """Returns a 0-1 score representing genuine depth in role-relevant skills."""
    skills = candidate.get("skills", [])
    sig = candidate.get("redrob_signals", {})
    raw_assessment = sig.get("skill_assessment_scores", {}) or {}
    # FIX: Normalise assessment keys to lowercase for case-insensitive lookup
    assessment_scores = {k.lower(): v for k, v in raw_assessment.items()}

    depth_total = 0.0
    relevant_count = 0

    for sk in skills:
        name_lower = sk.get("name", "").lower()
        if not any(ds in name_lower for ds in AI_DEPTH_SKILLS):
            continue

        relevant_count += 1
        proficiency = sk.get("proficiency", "")
        dur = sk.get("duration_months", 0) or 0   # FIX: guard None
        endorsements = sk.get("endorsements", 0) or 0  # FIX: guard None

        base = {"expert": 1.0, "advanced": 0.75, "intermediate": 0.4}.get(proficiency, 0.15)
        dur_mult = min(1.0, dur / 24.0) if dur > 0 else 0.1
        endorsement_bonus = min(0.3, math.log1p(endorsements) / 15.0)

        # FIX: Case-insensitive assessment score lookup + safe float cast
        assess_bonus = 0.0
        if name_lower in assessment_scores:
            assess_bonus = (_safe_float(assessment_scores[name_lower]) / 100.0) * 0.2

        depth_total += base * dur_mult + endorsement_bonus + assess_bonus

    if relevant_count == 0:
        return 0.0

    return min(1.0, depth_total / 10.0)  # normalised against ~10 strong skills as a ceiling


# Sanity check
for c in kept[:10]:
    s = score_skill_depth(c)
    print(f"{c['candidate_id']} | skill_depth={s:.3f} | title={c['profile']['current_title']}")


CAND_0000001 | skill_depth=0.211 | title=Backend Engineer
CAND_0000010 | skill_depth=0.026 | title=Data Engineer
CAND_0000011 | skill_depth=0.138 | title=QA Engineer
CAND_0000014 | skill_depth=0.115 | title=Frontend Engineer
CAND_0000015 | skill_depth=0.074 | title=Software Engineer
CAND_0000018 | skill_depth=0.000 | title=Frontend Engineer
CAND_0000021 | skill_depth=0.245 | title=Project Manager
CAND_0000023 | skill_depth=0.000 | title=Software Engineer
CAND_0000025 | skill_depth=0.101 | title=Frontend Engineer
CAND_0000027 | skill_depth=0.052 | title=DevOps Engineer


## 8. Behavioral Availability Score

This is the layer that uses Redrob's *platform activity signals* — the data that
exists precisely because Redrob has 790M+ profiles and ongoing engagement tracking,
something a static resume database (like a plain keyword filter) could never have.
The brief explicitly asks us to factor in "behavioral signals," and this is where we do it.

A candidate can be a perfect skill match on paper and still be a poor recruiting bet
if they're not actually reachable or interested. We compute:

- **Recency of activity** (up to 30 pts)
- **Open to work flag** (up to 20 pts)
- **Recruiter response rate** (up to 25 pts)
- **Interview completion rate** (up to 15 pts)
- **Notice period** (up to 10 pts)

In [8]:
def score_behavioral_availability(candidate: dict) -> float:
    """Returns a 0-1 score representing how engaged and reachable this candidate is."""
    sig = candidate.get("redrob_signals", {})
    score = 0.0

    last_active_s = sig.get("last_active_date", "")
    if last_active_s:
        try:
            last_active = datetime.strptime(last_active_s, "%Y-%m-%d").date()
            days_ago = (REFERENCE_DATE - last_active).days
            if days_ago <= 30:
                score += 0.30
            elif days_ago <= 90:
                score += 0.22
            elif days_ago <= 180:
                score += 0.12
            else:
                score += 0.02
        except ValueError:
            score += 0.05

    if sig.get("open_to_work_flag"):
        score += 0.20

    # FIX: Use _safe_float to handle None, strings, etc.
    score += _safe_float(sig.get("recruiter_response_rate", 0.0)) * 0.25
    score += _safe_float(sig.get("interview_completion_rate", 0.0)) * 0.15

    notice = sig.get("notice_period_days", 90)
    if notice <= 30:
        score += 0.10
    elif notice <= 60:
        score += 0.05
    else:
        score += 0.01

    return min(1.0, score)


for c in kept[:10]:
    s = score_behavioral_availability(c)
    sig = c["redrob_signals"]
    print(f"{c['candidate_id']} | behavioral={s:.3f} | "
          f"open_to_work={sig.get('open_to_work_flag')} | "
          f"response_rate={_safe_float(sig.get('recruiter_response_rate')):.2f} | "
          f"last_active={sig.get('last_active_date')}")


CAND_0000001 | behavioral=0.662 | open_to_work=True | response_rate=0.34 | last_active=2026-05-20
CAND_0000010 | behavioral=0.410 | open_to_work=False | response_rate=0.40 | last_active=2026-04-29
CAND_0000011 | behavioral=0.338 | open_to_work=False | response_rate=0.56 | last_active=2026-01-19
CAND_0000014 | behavioral=0.513 | open_to_work=False | response_rate=0.80 | last_active=2026-04-12
CAND_0000015 | behavioral=0.518 | open_to_work=True | response_rate=0.32 | last_active=2026-02-12
CAND_0000018 | behavioral=0.275 | open_to_work=False | response_rate=0.16 | last_active=2026-02-18
CAND_0000021 | behavioral=0.272 | open_to_work=False | response_rate=0.49 | last_active=2025-11-21
CAND_0000023 | behavioral=0.598 | open_to_work=False | response_rate=0.57 | last_active=2026-04-06
CAND_0000025 | behavioral=0.620 | open_to_work=True | response_rate=0.74 | last_active=2026-03-30
CAND_0000027 | behavioral=0.667 | open_to_work=True | response_rate=0.58 | last_active=2026-05-07


## 9. Career Trajectory Score

A candidate's *current* title only tells you where they are right now — it doesn't
tell you which *direction* they're moving in. This layer:

1. Sorts career history from most recent to oldest
2. Scores each role for AI/ML relevance
3. Weights recent roles more heavily (harmonic decay)
4. **Penalises pure research backgrounds** with no production signal
5. **Penalises short-tenure title-chasing** (avg < 12 months)
6. **Penalises consulting-only careers**

In [9]:
def score_career_trajectory(candidate: dict) -> float:
    """Returns a 0-1 score representing whether the candidate's career is moving
    toward (high score) or away from (low score) the target role."""
    career = candidate.get("career_history", [])
    if not career:
        return 0.3

    def sort_key(h):
        return "9999-99-99" if h.get("is_current") else h.get("start_date", "0000-00-00")

    sorted_career = sorted(career, key=sort_key, reverse=True)

    def role_relevance(hist):
        text = (hist.get("title", "") + " " + hist.get("description", "")).lower()
        ml_hits = sum(1 for ds in AI_DEPTH_SKILLS if ds in text)
        title_hits = sum(1 for gt in GOOD_TITLE_SIGNALS if gt in text)
        return min(1.0, ml_hits * 0.1 + title_hits * 0.2)

    role_scores = [role_relevance(h) for h in sorted_career]
    weights = [1.0 / (1.0 + 0.5 * i) for i in range(len(role_scores))]
    trajectory = sum(s * w for s, w in zip(role_scores, weights)) / sum(weights)

    # FIX: Production vs research signal check — use only descriptions (not company names)
    # to avoid false positives like "Infosys" -> substring match on "sys" -> "system".
    # Also use word boundaries for signal detection.
    all_text = " ".join(h.get("description", "") for h in career).lower()
    production_hits = sum(1 for p in [
        "production", "deployed", "shipped", "users", "traffic", "scale",
        "latency", "throughput", "api", "service", "system", "platform"
    ] if re.search(r'\b' + re.escape(p) + r'\b', all_text))
    research_hits = sum(1 for r in [
        "lab", "research", "paper", "academic", "thesis", "phd", "university",
        "professor", "publish"
    ] if re.search(r'\b' + re.escape(r) + r'\b', all_text))

    if research_hits > production_hits and production_hits < 3:
        trajectory *= 0.6  # discount pure-research profiles

    # Title-chasing discount
    if len(career) > 1:
        avg_duration = sum(h.get("duration_months", 12) for h in career) / len(career)
        if avg_duration < 12:
            trajectory *= 0.85

    # FIX: Secondary consulting-only check using word-boundary matching
    if all(_is_consulting_company(h.get("company", "")) for h in career):
        trajectory *= 0.5

    return min(1.0, trajectory)


for c in kept[:10]:
    s = score_career_trajectory(c)
    titles = [h["title"] for h in c["career_history"][:2]]
    print(f"{c['candidate_id']} | trajectory={s:.3f} | recent roles={titles}")


CAND_0000001 | trajectory=0.240 | recent roles=['Backend Engineer', 'Analytics Engineer']
CAND_0000010 | trajectory=0.300 | recent roles=['Data Engineer']
CAND_0000011 | trajectory=0.102 | recent roles=['QA Engineer', 'QA Engineer']
CAND_0000014 | trajectory=0.108 | recent roles=['Frontend Engineer', 'Software Engineer']
CAND_0000015 | trajectory=0.177 | recent roles=['Software Engineer', 'Mobile Developer']
CAND_0000018 | trajectory=0.046 | recent roles=['Frontend Engineer', 'Frontend Engineer']
CAND_0000021 | trajectory=0.000 | recent roles=['Project Manager', 'Marketing Manager']
CAND_0000023 | trajectory=0.200 | recent roles=['Software Engineer', 'Frontend Engineer']
CAND_0000025 | trajectory=0.154 | recent roles=['Frontend Engineer', 'Frontend Engineer']
CAND_0000027 | trajectory=0.030 | recent roles=['DevOps Engineer', 'DevOps Engineer']


## 10. JD-Specific Fit Score

This layer captures the remaining practical fit signals:

- **Years of experience band** (5–9 years ideal)
- **Location/relocation fit**
- **GitHub activity** (proxy for hands-on engagement)

In [10]:
PREFERRED_LOCATIONS = {"pune", "noida", "hyderabad", "mumbai", "delhi", "bangalore", "bengaluru", "india"}

def score_jd_specific(candidate: dict) -> float:
    """Returns a 0-1 score for practical JD-specific fit (experience band, location, GitHub)."""
    profile = candidate.get("profile", {})
    sig = candidate.get("redrob_signals", {})
    score = 0.0

    yoe = profile.get("years_of_experience", 0)
    if 5 <= yoe <= 9:
        score += 0.40
    elif 4 <= yoe < 5 or 9 < yoe <= 12:
        score += 0.25
    elif 3 <= yoe < 4 or 12 < yoe <= 15:
        score += 0.10
    else:
        score += 0.02

    loc = profile.get("location", "").lower()
    country = profile.get("country", "").lower()
    if any(city in loc for city in PREFERRED_LOCATIONS) or country == "india":
        score += 0.30
    elif sig.get("willing_to_relocate"):
        score += 0.20
    else:
        score += 0.05

    gh = sig.get("github_activity_score", -1)
    if gh is not None and gh >= 0:
        score += min(0.30, gh / 100.0 * 0.30)

    return min(1.0, score)


for c in kept[:10]:
    s = score_jd_specific(c)
    print(f"{c['candidate_id']} | jd_specific={s:.3f} | yoe={c['profile']['years_of_experience']} | loc={c['profile']['location']}")


CAND_0000001 | jd_specific=0.478 | yoe=6.9 | loc=Toronto
CAND_0000010 | jd_specific=0.401 | yoe=4.6 | loc=London
CAND_0000011 | jd_specific=0.417 | yoe=2.0 | loc=Hyderabad, Telangana
CAND_0000014 | jd_specific=0.700 | yoe=8.4 | loc=Hyderabad, Telangana
CAND_0000015 | jd_specific=0.700 | yoe=5.4 | loc=Trivandrum, Kerala
CAND_0000018 | jd_specific=0.700 | yoe=6.6 | loc=Bhubaneswar, Odisha
CAND_0000021 | jd_specific=0.419 | yoe=14.5 | loc=Bhubaneswar, Odisha
CAND_0000023 | jd_specific=0.295 | yoe=3.7 | loc=New York
CAND_0000025 | jd_specific=0.700 | yoe=7.3 | loc=Vizag, Andhra Pradesh
CAND_0000027 | jd_specific=0.516 | yoe=3.9 | loc=Kolkata, West Bengal


## 11. Composite Score — Combining All Layers

| Layer | Weight | Why this weight |
|---|---|---|
| Technical alignment (semantic) | 35% | Most important — actual experience match |
| Skill depth | 25% | Separates genuine expertise from keyword stuffing |
| Behavioral availability | 20% | Unresponsive candidates aren't useful |
| Career trajectory | 15% | Direction of growth matters |
| JD-specific fit | 5% | Tiebreaker signal |

In [11]:
WEIGHTS = {
    "technical": 0.35,
    "skill_depth": 0.25,
    "behavioral": 0.20,
    "trajectory": 0.15,
    "jd_specific": 0.05,
}

def composite_score(scores: dict) -> float:
    return sum(WEIGHTS[k] * scores[k] for k in WEIGHTS)

print("Composite weighting scheme:")
for k, v in WEIGHTS.items():
    print(f"  {k:12s} {v:.0%}")


Composite weighting scheme:
  technical    35%
  skill_depth  25%
  behavioral   20%
  trajectory   15%
  jd_specific  5%


## 12. Run the Full Pipeline on the Candidate Pool

Now we run every scoring layer across the filtered candidate pool in batches.

In [12]:
import time
start_time = time.time()

BATCH_SIZE = 256
all_results = []

for batch_start in range(0, len(kept), BATCH_SIZE):
    batch = kept[batch_start: batch_start + BATCH_SIZE]
    batch_texts = [build_candidate_text(c) for c in batch]
    technical_scores = score_technical_alignment_batch(batch_texts)

    for cand, tech_score in zip(batch, technical_scores):
        scores = {
            "technical":   float(tech_score),
            "skill_depth": score_skill_depth(cand),
            "behavioral":  score_behavioral_availability(cand),
            "trajectory":  score_career_trajectory(cand),
            "jd_specific": score_jd_specific(cand),
        }
        final = composite_score(scores)
        all_results.append((final, cand, scores))

    if batch_start % 5000 == 0:
        elapsed = time.time() - start_time
        print(f"  Scored {batch_start + len(batch):,}/{len(kept):,} | elapsed {elapsed:.1f}s")

elapsed = time.time() - start_time
print(f"\nScoring complete: {len(all_results):,} candidates scored in {elapsed:.1f} seconds")


  Scored 256/36,290 | elapsed 6.4s

Scoring complete: 36,290 candidates scored in 1027.0 seconds


## 13. Rank and Select the Top 100

Sort by composite score and normalise to 0.20–1.00 range.

In [13]:
all_results.sort(key=lambda x: -x[0])
top_100 = all_results[:100]

max_score = top_100[0][0]
min_score = top_100[-1][0]
score_range = max_score - min_score if max_score > min_score else 1.0

print(f"Top 100 selected. Raw score range: {min_score:.4f} to {max_score:.4f}")
print("\nTop 10 preview:")
for i, (raw_score, cand, scores) in enumerate(top_100[:10]):
    print(f"  #{i+1:3d} {cand['candidate_id']} | raw={raw_score:.4f} | "
          f"tech={scores['technical']:.2f} depth={scores['skill_depth']:.2f} "
          f"behav={scores['behavioral']:.2f} traj={scores['trajectory']:.2f}")


Top 100 selected. Raw score range: 0.5980 to 0.7815

Top 10 preview:
  #  1 CAND_0036184 | raw=0.7815 | tech=0.68 depth=1.00 behav=0.84 traj=0.56
  #  2 CAND_0011687 | raw=0.7621 | tech=0.58 depth=1.00 behav=0.86 traj=0.60
  #  3 CAND_0049896 | raw=0.7513 | tech=0.67 depth=0.99 behav=0.64 traj=0.70
  #  4 CAND_0018499 | raw=0.7453 | tech=0.56 depth=0.90 behav=0.79 traj=0.76
  #  5 CAND_0086022 | raw=0.7429 | tech=0.52 depth=1.00 behav=0.76 traj=0.74
  #  6 CAND_0009691 | raw=0.7334 | tech=0.64 depth=0.89 behav=0.68 traj=0.72
  #  7 CAND_0084819 | raw=0.7319 | tech=0.63 depth=0.98 behav=0.74 traj=0.54
  #  8 CAND_0029367 | raw=0.7311 | tech=0.62 depth=1.00 behav=0.71 traj=0.55
  #  9 CAND_0005649 | raw=0.7243 | tech=0.66 depth=0.93 behav=0.70 traj=0.49
  # 10 CAND_0051004 | raw=0.7196 | tech=0.60 depth=0.85 behav=0.77 traj=0.74


## 14. Evidence-Grounded Reasoning Generation

Every claim is traceable to a specific field in the JSON. No hallucination.

In [14]:
def generate_reasoning(candidate: dict, scores: dict, rank: int) -> str:
    profile = candidate.get("profile", {})
    skills = candidate.get("skills", [])
    sig = candidate.get("redrob_signals", {})

    title = profile.get("current_title", "Unknown")
    company = profile.get("current_company", "")
    yoe = profile.get("years_of_experience", 0)
    loc = profile.get("location", "")

    # FIX: Show advanced/expert skills first, then intermediate as secondary evidence
    top_skills = [
        s["name"] for s in skills
        if any(ds in s.get("name", "").lower() for ds in AI_DEPTH_SKILLS)
        and s.get("proficiency") in ("advanced", "expert")
    ][:4]
    if len(top_skills) < 4:
        intermediate_skills = [
            s["name"] for s in skills
            if any(ds in s.get("name", "").lower() for ds in AI_DEPTH_SKILLS)
            and s.get("proficiency") == "intermediate"
            and s["name"] not in top_skills
        ][:2]
    else:
        intermediate_skills = []
    relevant_skills = top_skills

    rr = _safe_float(sig.get("recruiter_response_rate", 0))
    open_work = sig.get("open_to_work_flag", False)
    notice = sig.get("notice_period_days", 90)
    gh = sig.get("github_activity_score", -1)

    parts = []
    company_str = f" at {company}" if company else ""
    parts.append(f"{title}{company_str}, {yoe:.1f} yrs exp, {loc}")

    # FIX: Include intermediate skills as "developing: X, Y"
    if relevant_skills and intermediate_skills:
        parts.append(f"relevant skills: {', '.join(relevant_skills)}; developing: {', '.join(intermediate_skills)}")
    elif relevant_skills:
        parts.append(f"relevant skills: {', '.join(relevant_skills)}")
    else:
        parts.append("limited directly relevant AI/ML skills")

    if open_work and rr >= 0.5:
        parts.append(f"actively engaged (open to work, {rr:.0%} response rate)")
    elif rr >= 0.6:
        parts.append(f"responsive ({rr:.0%} response rate)")
    elif rr < 0.15:
        parts.append(f"low engagement ({rr:.0%} response rate) — availability risk")

    if notice <= 30:
        parts.append("notice \u226430 days")
    elif notice > 60:
        parts.append(f"notice {notice} days — potential delay")

    if gh is not None and gh >= 60:
        parts.append(f"strong GitHub activity ({gh}/100)")

    # FIX: Guard against None title
    if rank > 50 and any(bt in (title or "").lower() for bt in BAD_PRIMARY_TITLES):
        parts.append(f"primary title ({title}) is non-technical — review skills claims carefully")

    reasoning = "; ".join(parts) + "."
    return reasoning[:250] if len(reasoning) <= 250 else reasoning[:247] + "..."


print("Sample reasoning for top 5 candidates:\n")
for i, (raw_score, cand, scores) in enumerate(top_100[:5]):
    reasoning = generate_reasoning(cand, scores, i + 1)
    print(f"#{i+1} {cand['candidate_id']}")
    print(f"   {reasoning}\n")


Sample reasoning for top 5 candidates:

#1 CAND_0036184
   Recommendation Systems Engineer at CRED, 6.0 yrs exp, Trivandrum, Kerala; relevant skills: FAISS, Hugging Face Transformers, LangChain, Semantic Search; actively engaged (open to work, 90% response rate); notice ≤30 days.

#2 CAND_0011687
   Senior NLP Engineer at Niramai, 7.8 yrs exp, Indore, Madhya Pradesh; relevant skills: TensorFlow, FAISS, Embeddings, LangChain; actively engaged (open to work, 89% response rate); notice ≤30 days; strong GitHub activity (76.3/100).

#3 CAND_0049896
   Search Engineer at Unacademy, 7.3 yrs exp, Vizag, Andhra Pradesh; relevant skills: LangChain, LlamaIndex, Sentence Transformers, PyTorch; notice 90 days — potential delay.

#4 CAND_0018499
   Senior Machine Learning Engineer at Zomato, 7.2 yrs exp, Noida, Uttar Pradesh; relevant skills: Weaviate, Recommendation Systems, scikit-learn, Pinecone; actively engaged (open to work, 61% response rate); notice ≤30 days; strong GitHub activity (...

#5 

## 15. Write the Final Submission File

In [15]:
output_rows = []
for i, (raw_score, cand, scores) in enumerate(top_100):
    rank = i + 1
    norm_score = 0.20 + 0.80 * (raw_score - min_score) / score_range
    # FIX: Reduced epsilon from 1e-6 to 1e-8 to avoid exceeding 4-decimal precision
    norm_score = round(norm_score - i * 1e-8, 6)
    reasoning = generate_reasoning(cand, scores, rank)
    output_rows.append({
        "candidate_id": cand["candidate_id"],
        "rank": rank,
        "score": round(norm_score, 4),
        "reasoning": reasoning,
    })

OUTPUT_PATH = "./submission.csv"
with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["candidate_id", "rank", "score", "reasoning"])
    writer.writeheader()
    writer.writerows(output_rows)

print(f"Submission written to {OUTPUT_PATH}")
print(f"Total rows: {len(output_rows)}")


Submission written to ./submission.csv
Total rows: 100


## 16. Self-Validation Checks Before Submitting

In [16]:
# pandas already imported in cell 1

df = pd.read_csv(OUTPUT_PATH)

print("Row count:", len(df), "(expected 100)")
print("Duplicate candidate_ids:", df["candidate_id"].duplicated().sum(), "(expected 0)")
print("Scores strictly decreasing:", (df["score"].diff().dropna() < 0).all())

# FIX: Use kept (not full candidates list) for honeypot cross-check — more memory efficient
final_ids = set(df["candidate_id"])
id_to_candidate = {c["candidate_id"]: c for c in kept}
honeypot_in_final = sum(1 for cid in final_ids if is_honeypot(id_to_candidate.get(cid, {})))
print(f"Honeypots in final top 100: {honeypot_in_final} ({honeypot_in_final/100:.1%}) — must be < 10%")

print("\nAll checks complete. Review submission.csv before final upload.")


Row count: 100 (expected 100)
Duplicate candidate_ids: 0 (expected 0)
Scores strictly decreasing: False
Honeypots in final top 100: 0 (0.0%) — must be < 10%

All checks complete. Review submission.csv before final upload.
